In [ ]:
# # Package installations for WhisperX with GPU support and speaker diarization (if needed)
# !apt update && apt install -y ffmpeg
# !pip install --upgrade pip

# # WhisperX (includes Faster-Whisper backend)
# !pip install git+https://github.com/m-bain/whisperX.git


# # PyTorch GPU version (if not already installed)

# !pip install torch==1.13.1 torchaudio==0.13.1
# !pip install pyannote.audio==2.1.1

# !pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118

# # Speaker diarization
# !pip install pyannote.audio

# # Optional: to save results as .docx
# !pip install python-docx
# pip install -r requirements.txt


In [ ]:
# import whisperx
from whisperx.diarize import DiarizationPipeline
from pyannote.audio import Pipeline
import torch
import os

# -----------------------------
# CONFIG
# -----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HF_TOKEN = "tokenetokentokengeeeeesemchaos"  # Replace with your Hugging Face token
NUM_FILES = 100
INPUT_FOLDER = "./"      # Folder with your audio files
OUTPUT_FOLDER = "./transcripts"  # Folder to save .txt transcripts

# Make sure output folder exists
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Structured initial prompt for maximal accuracy
INITIAL_PROMPT = """
متن‌های این فایل مربوط به روانشناسی، روانپزشکی و جامعه‌شناسی هستند. زبان اصلی فارسی است، اما ممکن است واژه‌های انگلیسی در متن وجود داشته باشد که باید دقیقاً همان‌طور نوشته شوند. 
لطفاً از املای استاندارد فارسی و اعداد فارسی استفاده کنید. 
توجه ویژه به اصطلاحات تخصصی و حوزه‌ای داشته باشید و آن‌ها را دقیق حفظ کنید. فهرست مهم‌ترین اصطلاحات و عبارات تخصصی:
دون خوان، هفت سنگ، DSM-5, CBT, PTSD, mindfulness, اضطراب، افسردگی، روان‌کاوی، روان‌درمانی، روان‌شناسی بالینی، روان‌شناسی کودک، اختلال دوقطبی، اختلال وسواسی، روان‌سنجی، مشاوره، درمان شناختی-رفتاری، روان‌تحلیلگری، استرس، هیجان، رفتاردرمانی، انگیزش
همه اصطلاحات تخصصی یا مشابه آن‌ها باید بدون تغییر و با دقت نوشته شوند. 
مدل باید تمام متن فارسی را روان و خوانا بنویسد و واژه‌های انگلیسی و اصطلاحات را به شکل اصلی خود حفظ کند. 
اگر تلفظ واژه‌ها مشابه واژه‌های فارسی یا انگلیسی باشد، همان واژه درست را حفظ کند.
"""

# -----------------------------
# LOAD MODELS
# -----------------------------
print("Loading WhisperX model...")
model = whisperx.load_model("large-v3", device=DEVICE, compute_type="float16")

print("Loading diarization pipeline...")
diarize_model = DiarizationPipeline(use_auth_token=HF_TOKEN, device=DEVICE)

# -----------------------------
# LOOP THROUGH AUDIO FILES
# -----------------------------
for i in range(1, NUM_FILES + 1):
    file_name = f"S({i}).mp3"
    file_path = os.path.join(INPUT_FOLDER, file_name)

    # Exit immediately if the first file is missing
    if i == 1 and not os.path.exists(file_path):
        raise FileNotFoundError(f"First file {file_name} not found. Exiting loop.")

    # Skip missing files after the first one
    if not os.path.exists(file_path):
        print(f"File {file_name} not found, skipping...")
        continue

    print(f"\n🔹 Processing {file_name}...")

    # Transcribe with initial prompt
    result = model.transcribe(
        file_path,
        initial_prompt=INITIAL_PROMPT,
        language="fa",  # Force Persian as main language
        beam_size=5,    # Improves transcription accuracy
        best_of=5
    )

    # Speaker diarization
    diarize_segments = diarize_model(file_path)

    # Assign speaker labels
    final_result = whisperx.assign_word_speakers(
        diarize_segments, result["segments"], fill_nearest=True
    )

    # Save transcript to separate .txt file
    base_name = os.path.splitext(file_name)[0]  # "S(1)"
    output_path = os.path.join(OUTPUT_FOLDER, f"{base_name}.txt")

    with open(output_path, "w", encoding="utf-8") as out_f:
        for seg in final_result["segments"]:
            speaker = seg.get("speaker", "Unknown")
            text = seg["text"]
            out_f.write(f"{speaker}: {text}\n")

    print(f"✅ Saved transcript to {output_path}")

print("\n🎉 All files processed!")


ModuleNotFoundError: No module named 'whisperx'